# Módulo 6: Big data con pyspark - Ejercicio de evaluación

In [2]:
# Importación de librerías necesarias
import pyspark
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, FloatType, StringType, IntegerType
from pyspark.sql.functions import col, sum
from pyspark.ml.feature import StringIndexer, Imputer, OneHotEncoder, VectorAssembler, MinMaxScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import requests

## 1. Carga de datos de diamonds desde CSV con schema:

In [3]:
# 1. Carga de datos de diamonds desde CSV con schema:

# Descargar y guardar el dataset localmente
dataset_url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv'
csv_path = 'diamonds.csv'
df_pandas = pd.read_csv(dataset_url)
df_pandas.to_csv(csv_path, index=False)

# Iniciar sesión de Spark
spark = SparkSession.builder.appName("DiamondsPipeline").getOrCreate()

# Definir el esquema explícito
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

schema = StructType([
    StructField("carat", DoubleType(), True),
    StructField("cut", StringType(), True),
    StructField("color", StringType(), True),
    StructField("clarity", StringType(), True),
    StructField("depth", DoubleType(), True),
    StructField("table", DoubleType(), True),
    StructField("price", DoubleType(), True),
    StructField("x", DoubleType(), True),
    StructField("y", DoubleType(), True),
    StructField("z", DoubleType(), True)
])

# Cargar datos con esquema explícito
df = spark.read.csv(csv_path, header=True, schema=schema)

## 2. Pipeline de Regresión para predecir 'price'

In [4]:
# 2. Pipeline de Regresión para predecir 'price'

imputer = Imputer(inputCols=["carat", "depth", "table", "x", "y", "z"], outputCols=["carat", "depth", "table", "x", "y", "z"])

# Indexar variables categóricas
indexers = [StringIndexer(inputCol=col, outputCol=col + "_index") for col in ["cut", "color", "clarity"]]

# One-Hot Encoding
encoders = [OneHotEncoder(inputCol=col + "_index", outputCol=col + "_ohe") for col in ["cut", "color", "clarity"]]

# Convertir `carat` en vector antes del escalado**
carat_assembler = VectorAssembler(inputCols=["carat"], outputCol="carat_vector")
scaler = MinMaxScaler(inputCol="carat_vector", outputCol="carat_scaled")

# VectorAssembler para todas las features
assembler = VectorAssembler(inputCols=["carat_scaled", "depth", "table", "x", "y", "z", "cut_ohe", "color_ohe", "clarity_ohe"], outputCol="features")

# Modelo de regresión
regressor = RandomForestRegressor(featuresCol="features", labelCol="price")

# Pipeline de regresión
reg_pipeline = Pipeline(stages=[imputer] + indexers + encoders + [carat_assembler, scaler, assembler, regressor])

# Entrenar modelo
reg_model = reg_pipeline.fit(df)
df_pred_reg = reg_model.transform(df)

# Evaluación regresión
evaluator_reg = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="r2")
r2 = evaluator_reg.evaluate(df_pred_reg)
print(f"Regresión R2: {r2}")

Regresión R2: 0.9065572677618794


## 3. Pipeline de Clasificación para predecir 'cut'

In [5]:
# 3. PIPELINE DE CLASIFICACIÓN SOBRE 'cut'
label_indexer = StringIndexer(inputCol="cut", outputCol="label")
assembler_clf = VectorAssembler(inputCols=["carat_scaled", "depth", "table", "x", "y", "z", "color_ohe", "clarity_ohe"], outputCol="features")
classifier = RandomForestClassifier(featuresCol="features", labelCol="label")
clf_pipeline = Pipeline(stages=[imputer, label_indexer] + indexers[1:] + encoders[1:] + [carat_assembler, scaler, assembler_clf, classifier])
clf_model = clf_pipeline.fit(df)
df_pred_clf = clf_model.transform(df)

# Evaluación clasificación
evaluator_clf = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator_clf.evaluate(df_pred_clf)
print(f"Clasificación Accuracy: {accuracy}")

Clasificación Accuracy: 0.6865962180200222


## 4. GridSearch con CrossValidation

In [6]:
# 4. VALIDACIÓN CRUZADA SOBRE REGRESIÓN
paramGrid = ParamGridBuilder().addGrid(regressor.numTrees, [10, 20]).build()
crossval = CrossValidator(estimator=reg_pipeline, estimatorParamMaps=paramGrid, evaluator=evaluator_reg, numFolds=3)
cv_model = crossval.fit(df)
cv_r2 = evaluator_reg.evaluate(cv_model.transform(df))
print(f"Cross-Validation R2: {cv_r2}")

Cross-Validation R2: 0.9065572677618794


### Experimental 1:

In [1]:
# Importación de librerías necesarias
import pyspark
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, FloatType, StringType, IntegerType
from pyspark.sql.functions import col, sum
from pyspark.ml.feature import StringIndexer, Imputer, OneHotEncoder, VectorAssembler, MinMaxScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import requests

# 1. Carga de datos de diamonds desde CSV con schema:

# Descargar y guardar el dataset localmente
dataset_url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv'
csv_path = 'diamonds.csv'
df_pandas = pd.read_csv(dataset_url)
df_pandas.to_csv(csv_path, index=False)

# Iniciar sesión de Spark
spark = SparkSession.builder.appName("DiamondsPipeline").getOrCreate()

# Definir el esquema explícito
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

schema = StructType([
    StructField("carat", DoubleType(), True),
    StructField("cut", StringType(), True),
    StructField("color", StringType(), True),
    StructField("clarity", StringType(), True),
    StructField("depth", DoubleType(), True),
    StructField("table", DoubleType(), True),
    StructField("price", DoubleType(), True),
    StructField("x", DoubleType(), True),
    StructField("y", DoubleType(), True),
    StructField("z", DoubleType(), True)
])

# Cargar datos con esquema explícito
df = spark.read.csv(csv_path, header=True, schema=schema)

# 2. Pipeline de Regresión para predecir 'price'

imputer = Imputer(inputCols=["carat", "depth", "table", "x", "y", "z"], outputCols=["carat", "depth", "table", "x", "y", "z"])

# Indexar variables categóricas
indexers = [StringIndexer(inputCol=col, outputCol=col + "_index") for col in ["cut", "color", "clarity"]]

# One-Hot Encoding
encoders = [OneHotEncoder(inputCol=col + "_index", outputCol=col + "_ohe") for col in ["cut", "color", "clarity"]]

# Convertir `carat` en vector antes del escalado**
carat_assembler = VectorAssembler(inputCols=["carat"], outputCol="carat_vector")
scaler = MinMaxScaler(inputCol="carat_vector", outputCol="carat_scaled")

# VectorAssembler para todas las features
assembler = VectorAssembler(inputCols=["carat_scaled", "depth", "table", "x", "y", "z", "cut_ohe", "color_ohe", "clarity_ohe"], outputCol="features")

# Modelo de regresión
regressor = RandomForestRegressor(featuresCol="features", labelCol="price")

# Pipeline de regresión
reg_pipeline = Pipeline(stages=[imputer] + indexers + encoders + [carat_assembler, scaler, assembler, regressor])

# Entrenar modelo
reg_model = reg_pipeline.fit(df)
df_pred_reg = reg_model.transform(df)

# Evaluación regresión
evaluator_reg = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="r2")
r2 = evaluator_reg.evaluate(df_pred_reg)
print(f"Regresión R2: {r2}")

# 3. PIPELINE DE CLASIFICACIÓN SOBRE 'cut'
label_indexer = StringIndexer(inputCol="cut", outputCol="label")
assembler_clf = VectorAssembler(inputCols=["carat_scaled", "depth", "table", "x", "y", "z", "color_ohe", "clarity_ohe"], outputCol="features")
classifier = RandomForestClassifier(featuresCol="features", labelCol="label")
clf_pipeline = Pipeline(stages=[imputer, label_indexer] + indexers[1:] + encoders[1:] + [carat_assembler, scaler, assembler_clf, classifier])
clf_model = clf_pipeline.fit(df)
df_pred_clf = clf_model.transform(df)

# Evaluación clasificación
evaluator_clf = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator_clf.evaluate(df_pred_clf)
print(f"Clasificación Accuracy: {accuracy}")

# 4. VALIDACIÓN CRUZADA SOBRE REGRESIÓN
paramGrid = ParamGridBuilder().addGrid(regressor.numTrees, [10, 20]).build()
crossval = CrossValidator(estimator=reg_pipeline, estimatorParamMaps=paramGrid, evaluator=evaluator_reg, numFolds=3)
cv_model = crossval.fit(df)
cv_r2 = evaluator_reg.evaluate(cv_model.transform(df))
print(f"Cross-Validation R2: {cv_r2}")

Regresión R2: 0.9065572677618794
Clasificación Accuracy: 0.6865962180200222
Cross-Validation R2: 0.9065572677618794
